In [ ]:
%run ./01-config

In [ ]:
class SetupHelper:
    def __init__(self, env):
        Conf = Config()
        self.landing_zone = Conf.base_dir_data + '/raw'
        self.checkpoint_base = Conf.base_dir_checkpoint + '/checkpoints'
        self.catalog = env
        self.db_name = Conf.db_name
        self.initialized = False

    def create_db(self):
        import time as ti
        st = ti.time()
        spark.sql('Clear Cache')
        print(f'Creating database {self.catalog}.{self.db_name}......', end = '')
        spark.sql(f'create database if not exists {self.catalog}.{self.db_name}')
        spark.sql(f'use {self.catalog}.{self.db_name}')
        self.initalized = True
        print('Done\n')
        print(f'Completed in {round(ti.time() - st, 2)}')

    def create_registered_users_bz(self): # creating bronze table hence using create table if not exists
                                          # since its bronze its also has load_time and source_file as columns
        if self.initalized:
            print('Creating registered_users_bz table...', end = '')
            spark.sql(f'''create table if not exists {self.catalog}.{self.db_name}.registered_users_bz
                        (user_id string,
                        device_id long,
                        mac_address string,
                        registration_timestamp double,
                        load_time timestamp,
                        source_file string)
                        ''')
            print('Done')
        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_gym_logins_bz(self): # creating bronze table hence using create table if not exists
                                 # since its bronze its also has load_time and source_file as columns
        if self.initalized:
            print('Creating gym_log_bz table...', end = '')
            spark.sql(f'''create table if not exists {self.catalog}.{self.db_name}.gym_logins_bz
                      (mac_address string,
                      gym bigint,
                      login double,
                      logout double,
                      load_time timestamp,
                      source_file string
                      )
                      ''')
            print('Done')
        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_kafka_multiplex_bz(self): # creating bronze table hence using create table if not exists
                                         # since its bronze its also has load_time and source_file as columns
        if self.initalized:
            print('Creating kafka_multiplex_bz table...', end = '')
            spark.sql(f'''create table if not  exists {self.catalog}.{self.db_name}.kafka_multiplex_bz
                      (key string,
                      value string,
                      topic string,
                      partition bigint,
                      offset bigint,
                      timestamp bigint,
                      date date,
                      week_part string,
                      load_time timestamp,
                      source_file string)
                      partitioned by (topic, week_part)
                      ''')
            print('Done')
        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_users(self):# creating silver table hence using create or replace
                           # it references registered_users_bz bronze table
        if self.initalized:
            print('Creating users table...', end = '')
            spark.sql(f'''create or replace table {self.catalog}.{self.db_name}.users
                      (
                       user_id string,
                       device_id long,
                       mac_address string,
                       registration_timestamp double  
                      )
                      ''')
            print('Done')

        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_gym_logs(self):# creating silver table hence using create or replace
                              # it references gym_logs_bz bronze table
        if self.initalized:
            print('Creating gym_logs table...', end = '')
            spark.sql(f'''create or replace table {self.catalog}.{self.db_name}.gym_logs
                      (
                      mac_address string,
                      gym string,
                      login timestamp,
                      logout timestamp
                      )
                      ''')
            print('Done')
        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

        

    def create_user_profile(self): # creating silver table hence using create or replace 
                                   # it references kafaka_multiplex_bz bronze table
        if self.initalized:
            print('Creating user_profile table...', end = '')
            spark.sql(f'''create or replace table {self.catalog}.{self.db_name}.user_profile
                      (user_id string,
                      dob date,
                      sex string,
                      gender string,
                      first_name string,
                      last_name string,
                      street_address string,
                      city string,
                      state string,
                      zip int,
                      updated timestamp)
                      ''')
            print('Done')
        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_heart_rate(self):# creating silver table hence using create or replace 
                                # it references kafaka_multiplex_bz bronze table
        if self.initalized:
            print('Creating heart_rate table...', end = '')
            spark.sql(f'''create or replace table {self.catalog}.{self.db_name}.heart_rate
                      (device_id long,
                      time timestamp,
                      heartrate double,
                      valid boolean)
                      ''')
            print('Done')

        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')


    def create_workouts (self):
        if self.initalized:
            print('Creating workouts table...', end = '')
            spark.sql(f'''create or replace table {self.catalog}.{self.db_name}.workouts
                      (user_id long,
                      time timestamp,
                      workout_id bigint,
                      action string,
                      session_id int)
                      ''')
            print('Done')

        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_completed_workouts(self):
        if self.initalized:
            print('Creating completed_workouts table...', end = '')
            spark.sql(f'''create or replace table {self.catalog}.{self.db_name}.completed_workouts
                      (user_id string,
                      workout_id string,
                      session_id string,
                      start_time timestamp,
                      end_time timestamp
                      )
                      ''')
            print('Done')
        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_user_bins(self):
        if self.initalized:
            print('Creating user_bins table...', end = '')
            spark.sql(f'''create table if not exists {self.catalog}.{self.db_name}.user_bins
                      (user_id string,
                      age string,
                      gender string,
                      city string,
                      state string
                      )
                      ''')
            print('Done')
        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_workout_bpm(self):
        if self.initalized:
            print('Creating workout table...', end = '')
            spark.sql(f'''create or replace table {self.catalog}.{self.db_name}.workout_bpm
                      (user_id int,
                      workout_id int,
                      time timestamp,
                      session_id string,
                      start_time timestamp,
                      end_time timestamp,
                      heartrate double
                      )''')
            print('Done')
        else:
            raise ReferecenceError('Application database is not defined. Cannot create table in default database')

    def create_date_lookup(self):
        if self.initalized:
            print('Creating date_lookup table...', end = '')
            spark.sql(f'''create or replace table {self.catalog}.{self.db_name}.date_lookup
                      (date date,
                      week int,
                      year int,
                      month int,
                      dayofweek int,
                      dayofmonth int,
                      dayofyear int,
                      week_part string
                      )
                      ''')
            print('Done')
        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_workout_bpm_summary(self):
        if self.initalized:
            print('Creating workout_bpm_summary table...', end = '')
            spark.sql(f'''create table if not exists {self.catalog}.{self.db_name}.workout_bpm_summary
                      (workout_id int,
                      session_id int,
                      user_id int,
                      age string,
                      gender string,
                      city string,
                      state string,
                      min_bpm double,
                      max_bpm double,
                      num_recordings bigint
                      )
                      ''')
            print('Done')
        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def create_gym_summary(self):
        if self.initalized:
            print('Creating gym_summary table...', end = '')
            spark.sql(f'''create or replace view {self.catalog}.{self.db_name}.gym_summary as
                      select to_date(login::timestamp) date,
                      gym, l.mac_address, workout_id, session_id,
                      round((logout::long - login::long)/60,2) minutes_in_gym,
                      round((end_time::long - start_time::long)/60,2) minutes_exercising
                      from gym_logs l
                      join (
                        select mac_address, workout_id, session_id, start_time, end_time
                        from completed_workouts w
                        inner join users u on w.user_id = u.user_id) w
                      on l.mac_address = w.mac_address
                      and w.start_time between l.login and l.logout
                      order by date, gym, l.mac_address, session_id                      
                      ''')
            print('Done')

        else:
            raise ReferenceError('Application database is not defined. Cannot create table in default database')

    def setup(self):
        import time
        st = time.time()
        print(f'\nStarting Setup\n')
        self.create_db()
        self.create_registered_users_bz()
        self.create_gym_logins_bz()
        self.create_kafka_multiplex_bz()
        self.create_users()
        self.create_gym_logs()
        self.create_user_profile()
        self.create_heart_rate()
        self.create_workouts()
        self.create_completed_workouts()
        self.create_user_bins()
        self.create_workout_bpm()
        self.create_date_lookup()
        self.create_workout_bpm_summary()
        self.create_gym_summary()
        print(f"Setup completed in {round(int(time.time()) - st,2)} seconds")
    
    def assert_table(self, table_name):
        from pyspark.sql import functions as f 
        assert (spark.sql(f'''show tables in {self.catalog}.{self.db_name}''')
                        .where((f.col('isTemporary') == 'false') & (f.col('tableName') == f'{table_name}'))
                        .count()) == 1, f'The table {table_name} is missing'
        print(f'Found {table_name} in {self.catalog}.{self.db_name}: Success')

    def validate(self):
        from pyspark.sql import functions as f 
        import time
        start = time.time()

        assert (spark.sql(f'show databases in {self.catalog}')
                .where(f.col('databaseName') == self.db_name)
                .count()) == 1, f'The database {self.db_name} is missing in the catalog {self.catalog}'
        print(f"Found database {self.catalog}.{self.db_name}: Success")

        self.assert_table('registered_users_bz')
        self.assert_table('gym_logins_bz')
        self.assert_table('kafka_multiplex_bz')
        self.assert_table('users')
        self.assert_table('gym_logs')
        self.assert_table('user_profile')
        self.assert_table('heart_rate')
        self.assert_table('workouts')
        self.assert_table('completed_workouts')
        self.assert_table('user_bins')
        self.assert_table('workout_bpm')
        self.assert_table('date_lookup')
        self.assert_table('workout_bpm_summary')
        self.assert_table('gym_summary')
        print(f"Setup validation completed in {round (time.time() - start, 2)} seconds")

    def cleanup(self):
        from pyspark.sql import functions as f
        if (spark.sql(f'show databases in {self.catalog}')\
            .where(f.col('databaseName') == self.db_name)
            .count()) == 1:
                print(f"Dropping the database {self.catalog}.{self.db_name}...", end='')
                spark.sql(f'drop database {self.catalog}.{self.db_name} cascade')
                print('Done')

        print(f'Deleting landing zone {self.landing_zone}......', end='' )
        dbutils.fs.rm(self.landing_zone, True)
        print('Done')
        print(f'Deleting chekpoint zone {self.checkpoint_base}.......', end = '')
        dbutils.fs.rm(self.checkpoint_base, True)
        print('Done')